# 04x -- Manual Dynasty Rankings (DynastySharks + FantasyPros)

**Purpose:** Ingest hand-extracted dynasty ranking sheets into the section-04
two-layer model. Each sheet = one (source, format). Sources expose different
beyond-ranking metrics — all land as `metric_key` rows, so the schema never changes.

**Input:** `data/raw/DynastyRankings_2026_ManualExtraction.xlsx`
| sheet | source_name | format | extra metrics |
|---|---|---|---|
| `SF PPR-DynastySharks` | DynastySharks | SF | 1/3/5/10-yr proj, 3D value, adp, analysis(text) |
| `SF TE Premium-DynastySharks` | DynastySharks | TEPP | (same) |
| `SF PPR-FantasyPros` | FantasyPros | SF | best / worst / avg / stddev |
| `IDP-FantasyPros` | FantasyPros | IDP | best / worst / avg / stddev |

**Outputs:** appends to `fact_dynasty_rankings` + `fact_dynasty_ranking_metrics`
(replace-by-`(snapshot_date, source_name)`); upserts `dim_dynasty_crosswalk` per
source via the shared `etl.resolve_dynasty_crosswalk`. Run with CWD = repo root.

In [ ]:
# ---- Setup & Config ---------------------------------------------------------
import re
from dataclasses import dataclass, field
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

import sys
for _p in (Path.cwd() / "notebooks", Path.cwd()):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG


@dataclass
class ManualDynastyConfig:
    """Manual dynasty-ranking ingest config."""
    xlsx: str = "data/raw/DynastyRankings_2026_ManualExtraction.xlsx"
    # sheet -> (source_name, format)
    sheets: dict = field(default_factory=lambda: {
        "SF PPR-DynastySharks":        ("DynastySharks", "SF"),
        "SF TE Premium-DynastySharks": ("DynastySharks", "TEPP"),
        "SF PPR-FantasyPros":          ("FantasyPros",   "SF"),
        "IDP-FantasyPros":             ("FantasyPros",   "IDP"),
    })
    # per-source numeric metric_key -> column header
    metric_maps: dict = field(default_factory=lambda: {
        "DynastySharks": {
            "adp": "ADP", "proj_1yr": "1yr. Proj", "proj_3yr": "3yr. Proj",
            "proj_5yr": "5yr. Proj", "proj_10yr": "10yr. Proj", "ds_value": "3D Value +",
        },
        "FantasyPros": {
            "best": "BEST", "worst": "WORST", "avg": "AVG.", "stddev": "STD.DEV",
        },
    })
    text_maps: dict = field(default_factory=lambda: {
        "DynastySharks": {"analysis": "DS Analysis"},
        "FantasyPros": {},
    })
    crosswalk: str = "dim_dynasty_crosswalk"
    fact_rank: str = "fact_dynasty_rankings"
    fact_metrics: str = "fact_dynasty_ranking_metrics"


MCFG = ManualDynastyConfig()
SNAPSHOT_DATE = date.today().isoformat()


def _slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", str(name).lower()).strip("-")

def _num(v):
    """Coerce a sheet cell to float, or None ('-', '', NaN, text)."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    s = str(v).replace(",", "").replace("%", "").strip()
    if s in ("", "-"):
        return None
    try:
        return float(s)
    except ValueError:
        return None

def _split_pos(tok):
    """'QB1' -> ('QB', 1); 'DE2' -> ('DE', 2); 'QB' -> ('QB', None)."""
    m = re.match(r"\s*([A-Za-z/]+)\s*(\d*)", str(tok))
    if not m:
        return str(tok), None
    return m.group(1), (int(m.group(2)) if m.group(2) else None)

print(f"snapshot_date={SNAPSHOT_DATE} | sheets={len(MCFG.sheets)}")

In [ ]:
# ---- Parse sheets -> backbone rows + metric rows ----------------------------
xl = pd.ExcelFile(MCFG.xlsx)
backbone, metrics = [], []

for sheet, (source_name, fmt) in MCFG.sheets.items():
    df = pd.read_excel(MCFG.xlsx, sheet_name=sheet)
    age_col = "Age" if "Age" in df.columns else ("AGE" if "AGE" in df.columns else None)
    mmap = MCFG.metric_maps.get(source_name, {})
    tmap = MCFG.text_maps.get(source_name, {})

    for row in df.to_dict("records"):   # to_dict preserves non-identifier headers
        name = row.get("Name")
        if not isinstance(name, str) or not name.strip():
            continue
        pos_raw, pos_rank = _split_pos(row.get("Pos"))
        spid = _slug(name)
        age = _num(row.get(age_col)) if age_col else None
        rk = _num(row.get("RK"))
        backbone.append({
            "snapshot_date": SNAPSHOT_DATE, "source_name": source_name,
            "source_player_id": spid, "format": fmt, "source_site": source_name,
            "player_name": name, "position_raw": pos_raw,
            "nfl_team": row.get("Team"),
            "age": age,
            "overall_rank": int(rk) if rk is not None else None,
            "positional_rank": pos_rank,
        })
        for mk, col in mmap.items():
            num = _num(row.get(col))
            if num is not None:
                metrics.append({"snapshot_date": SNAPSHOT_DATE, "source_name": source_name,
                                "source_player_id": spid, "format": fmt,
                                "metric_key": mk, "metric_num": num, "metric_text": None})
        for mk, col in tmap.items():
            v = row.get(col)
            if isinstance(v, str) and v.strip():
                metrics.append({"snapshot_date": SNAPSHOT_DATE, "source_name": source_name,
                                "source_player_id": spid, "format": fmt,
                                "metric_key": mk, "metric_num": None, "metric_text": v.strip()})

backbone = pd.DataFrame(backbone)
metrics = pd.DataFrame(metrics)
print("backbone rows by (source, format):")
print(backbone.groupby(["source_name", "format"]).size().to_string())
print(f"metric rows: {len(metrics)} | keys: {sorted(metrics['metric_key'].unique())}")

In [ ]:
# ---- Resolve identity per source (shared matcher) + upsert crosswalk --------
xpath = f"{CFG.data_dir}/{MCFG.crosswalk}.parquet"
xwalk = pd.read_parquet(xpath) if Path(xpath).exists() else pd.DataFrame()
all_review = []

# Manual gsis overrides for nickname/full-name mismatches (fuzzy < 90):
MANUAL_OVERRIDES = {
    "DynastySharks": {
        "cameron-ward":      "00-0040676",   # -> Cam Ward (QB)
        "cameron-skattebo":  "00-0040715",   # -> Cam Skattebo (RB)
        "chigoziem-okonkwo": "00-0037809",   # -> Chig Okonkwo (TE)
    },
    "FantasyPros": {
        "hollywood-brown":   "00-0035662",   # -> Marquise Brown (WR)
        "matt-hibner":       "HIB564972",    # -> Matthew Hibner (TE)
        "foyesade-oluokun":  "00-0034413",   # -> Foye Oluokun (LB)
        "cam-bynum":         "00-0036629",   # -> Camryn Bynum (S)
    },
}   # Daylan Smothers / Mark Fletcher: 2026 rookies absent from both registries -> stay unmatched


for source_name in backbone["source_name"].unique():
    ident = (backbone[backbone["source_name"] == source_name]
             [["source_player_id", "player_name", "position_raw", "nfl_team"]]
             .drop_duplicates("source_player_id").copy())
    ident["source"] = source_name
    res = etl.resolve_dynasty_crosswalk(ident, data_dir=CFG.data_dir,
                                        overrides=MANUAL_OVERRIDES.get(source_name))
    print(f"\n{source_name}: {len(res)} ids")
    print(res["match_method"].value_counts().to_string())
    if len(xwalk):
        xwalk = xwalk[xwalk["source"] != source_name]
    xwalk = pd.concat([xwalk, res], ignore_index=True)
    rv = res[res["match_method"].isin(["review", "unmatched"])]
    if len(rv):
        all_review.append(rv)

xwalk.to_parquet(xpath, index=False)
print(f"\n[ok] crosswalk now {len(xwalk)} rows across {xwalk['source'].nunique()} sources -> {xpath}")

# Review file = projection of the FULL crosswalk's unresolved rows (all sources),
# rebuilt each run so resolved rows drop out (not an accumulator).
rvp = Path(CFG.data_dir) / "review" / "review_dynasty_crosswalk.csv"
rvp.parent.mkdir(parents=True, exist_ok=True)
unresolved = xwalk[xwalk["match_method"].isin(["review", "unmatched"])].copy()
if len(unresolved):
    unresolved["action"] = ""          # user fills: a gsis_id, or "new"
    unresolved.to_csv(rvp, index=False)
    print(f"[review] {len(unresolved)} unresolved across all sources -> {rvp}")
else:
    if rvp.exists():
        rvp.unlink()
    print("[review] none unresolved")

In [ ]:
# ---- Attach FKs + load both facts (replace-by-(snapshot_date, source_name)) --
fk = xwalk.set_index(["source", "source_player_id"])[["gsis_id", "player_key"]]
def _fk(row, col):
    try:
        return fk.loc[(row["source_name"], row["source_player_id"]), col]
    except KeyError:
        return None
backbone["gsis_id"]    = backbone.apply(lambda r: _fk(r, "gsis_id"), axis=1)
backbone["player_key"] = backbone.apply(lambda r: _fk(r, "player_key"), axis=1)

backbone = backbone[["snapshot_date", "source_name", "source_player_id", "format",
                     "source_site", "player_name", "position_raw", "nfl_team", "age",
                     "overall_rank", "positional_rank", "gsis_id", "player_key"]]

def load_partition(new, name, parts=("snapshot_date", "source_name")):
    path = f"{CFG.data_dir}/{name}.parquet"
    parts = list(parts)
    if Path(path).exists():
        old = pd.read_parquet(path)
        keys = set(map(tuple, new[parts].drop_duplicates().to_numpy()))
        mask = old[parts].apply(tuple, axis=1).isin(keys)
        out = pd.concat([old[~mask], new], ignore_index=True)
    else:
        out = new
    out.to_parquet(path, index=False)
    return path, len(out)

# Single-column surrogate for PBI relationships (source_player_id alone is not unique).
backbone["source_uid"] = backbone["source_name"] + "|" + backbone["source_player_id"]
metrics["source_uid"]  = metrics["source_name"] + "|" + metrics["source_player_id"]
bp, bn = load_partition(backbone, MCFG.fact_rank)
mp, mn = load_partition(metrics, MCFG.fact_metrics)
print(f"[ok] +{len(backbone)} backbone rows (table now {bn}) -> {bp}")
print(f"[ok] +{len(metrics)} metric rows (table now {mn}) -> {mp}")

In [ ]:
# ---- Validation -------------------------------------------------------------
bb = pd.read_parquet(f"{CFG.data_dir}/{MCFG.fact_rank}.parquet")
mt = pd.read_parquet(f"{CFG.data_dir}/{MCFG.fact_metrics}.parquet")

print("=== fact_dynasty_rankings by source x format ===")
print(bb.groupby(["source_name", "format"]).size().to_string())
print(f"\ntotal backbone rows: {len(bb)}")
dups = bb.duplicated(["snapshot_date","source_name","source_player_id","format"]).sum()
print(f"backbone key dupes: {dups} (expect 0)")
this = bb[bb["source_name"].isin(["DynastySharks","FantasyPros"])]
print(f"manual-source gsis coverage: {this['gsis_id'].notna().mean()*100:.1f}% "
      f"| player_key: {this['player_key'].notna().mean()*100:.1f}%")

print("\n=== metric keys present ===")
print(mt.groupby(["source_name","metric_key"]).size().to_string())

print("\n=== DynastySharks SF top 5 ===")
ds = bb[(bb["source_name"]=="DynastySharks") & (bb["format"]=="SF")].sort_values("overall_rank")
print(ds[["overall_rank","positional_rank","player_name","position_raw","age","gsis_id"]].head().to_string(index=False))